# RuO2/TiO2 crystal truncation rods in DWBA

This notebook is the DWBA counterpart of `RuO2_TiO2_CTR_test.ipynb`. It loads the same TiO2 bulk / graded epitaxy interface / RuO2 film / Poisson surface model. On the specular rod it compares the total DWBA field with the kinematical structure factor after converting between their normalizations; off specular it compares the coherent DWBA contrast factor directly.

Two rods are evaluated at 20 keV:

- the specular `(0, 0, L)` rod with equal incident and exit angles;
- the measured non-specular `(0, 1, L)` rod with the incident angle fixed at $1.5\alpha_c$.

On the specular rod, `F_contrast = F_atomic - F_reference` is only the correction. The physical amplitude is `total_amplitude = r0 + scattered_amplitude`. To compare it with the kinematical structure factor in electron units, the notebook divides the total field by the same point-dependent scattering prefactor used by `DWBAResult`. Off specular, `F_reference` and `r0` are exactly zero. The serialized `atten=0.01` remains in the kinematical curve, while DWBA uses the absorption in the complex refractive indices and therefore keeps its default `bulk_attenuation=0`.

In [ ]:
%matplotlib widget
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np


def find_repository_root():
    """Find the checkout containing the current example notebook."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "orgui" / "datautils").is_dir():
            return candidate
    return None


repository_root = find_repository_root()
if repository_root is not None and "orgui" not in sys.modules:
    sys.path.insert(0, str(repository_root))

from orgui.datautils.xrayutils import CTRcalc, CTRuc
from orgui.datautils.xrayutils.CTRoptics import homogeneous_bulk_profile


def find_example_file(filename):
    """Find an example file from the checkout or notebook directory."""
    candidates = (Path(filename), Path("examples/CTR") / filename)
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(filename)


if not hasattr(CTRcalc.SXRDCrystal, "dwba"):
    raise RuntimeError("The loaded orGUI build does not provide DWBA.")
if not hasattr(CTRuc._CTRcalc_cpp, "unitcell_F_DWBA_records"):
    raise RuntimeError(
        "Rebuild the native extension before running this notebook."
    )

## Model and critical angle

The substrate dispersion defines the critical angle in the small-angle X-ray approximation, $\alpha_c=\sqrt{2\delta_{\mathrm{TiO_2}}}$. The reference profile used here is the same homogeneous bulk profile that anchors the graded DWBA optical stack.

In [ ]:
model_path = find_example_file("RuO2_TiO2_Poisson_etching.xtal")
crystal = CTRcalc.SXRDCrystal.fromFile(model_path)

bulk_profile = homogeneous_bulk_profile(crystal.uc_bulk)
delta_bulk = float(bulk_profile.values[0, 1])
alpha_c = float(np.sqrt(2.0 * delta_bulk))
alpha_i_fixed = 1.5 * alpha_c

print(f"model: {model_path}")
print(f"energy: {crystal.uc_bulk._E * 1e-3:g} keV")
print(f"TiO2 delta: {delta_bulk:.6g}")
print(f"alpha_c: {np.rad2deg(alpha_c):.5f} deg")
print(f"1.5 alpha_c: {np.rad2deg(alpha_i_fixed):.5f} deg")
print(f"kinematical attenuation: {crystal.atten:g}")
print("components:", crystal.getUcNames())
crystal

## Specular and non-specular rods

`set_ctr_geometry` stores the experimental constraint for each rod. The non-specular lower limit is chosen so that the derived exit glancing angle remains positive. Off specular, both `crystal.F` and `result.F_contrast` are expressed in electrons per reference surface cell. Specularly, the comparable electron-unit quantity is obtained from `result.total_amplitude` by inverting the field normalization.

In [ ]:
L_specular = np.linspace(0.02, 7.0, 4001)
h_specular = np.zeros_like(L_specular)
k_specular = np.zeros_like(L_specular)

crystal.dwba.set_ctr_geometry(
    equal_angles=True, rods=[(0.0, 0.0)]
)
specular = crystal.dwba.evaluate(
    h_specular, k_specular, L_specular
)
F_kinematic_specular = crystal.F(
    h_specular, k_specular, L_specular
)

CLASSICAL_ELECTRON_RADIUS_ANGSTROM = 2.8179403262e-5


def structure_factor_to_field(F, prepared):
    """Convert an electron-unit structure factor to field amplitude."""
    return (
        2j * np.pi * CLASSICAL_ELECTRON_RADIUS_ANGSTROM * F
        / (
            prepared.k0 * np.sin(prepared.alpha_f)
            * prepared.reference_area
        )
    )


def field_to_structure_factor(amplitude, prepared):
    """Express a field amplitude in equivalent electron units."""
    return (
        amplitude * prepared.k0 * np.sin(prepared.alpha_f)
        * prepared.reference_area
        / (2j * np.pi * CLASSICAL_ELECTRON_RADIUS_ANGSTROM)
    )


F_dwba_total_specular = field_to_structure_factor(
    specular.total_amplitude, specular.prepared
)
r_kinematic_specular = structure_factor_to_field(
    F_kinematic_specular, specular.prepared
)
np.testing.assert_allclose(
    field_to_structure_factor(
        specular.scattered_amplitude, specular.prepared
    ),
    specular.F_contrast,
)

L_rod = np.linspace(0.06, 6.0, 4001)
h_rod = np.zeros_like(L_rod)
k_rod = np.ones_like(L_rod)

crystal.dwba.set_ctr_geometry(
    alpha_i=alpha_i_fixed, rods=[(0.0, 1.0)]
)
non_specular = crystal.dwba.evaluate(h_rod, k_rod, L_rod)
F_kinematic_rod = crystal.F(h_rod, k_rod, L_rod)

assert np.all(specular.prepared.is_specular)
assert not np.any(non_specular.prepared.is_specular)
np.testing.assert_allclose(non_specular.F_reference, 0.0)
np.testing.assert_allclose(
    sum(
        (item.F_contrast for item in non_specular.contributions),
        start=0j,
    ),
    non_specular.F_contrast,
)

print("DWBA contribution records")
for item in specular.contributions:
    layer = "" if item.layer is None else f" layer={item.layer:g}"
    print(
        f"  {item.component_name:16s} {item.role:22s}"
        f" {item.record_name}{layer}"
    )
print(
    "non-specular alpha_f range: "
    f"{np.rad2deg(non_specular.prepared.alpha_f[0]):.4f} to "
    f"{np.rad2deg(non_specular.prepared.alpha_f[-1]):.4f} deg"
)

## DWBA versus kinematical scattering

The upper panels compare like-normalized electron-unit amplitudes. Specularly, the DWBA curve is the total field $r_0+\delta r$ expressed as an equivalent structure factor; off specular it is $F_\Delta$, because no unperturbed reflection is present. The lower-left panel shows the same specular comparison directly in dimensionless field-intensity units.

In [ ]:
def plot_factor_comparison(
    ax, L, F_dwba, F_kinematic, title, dwba_label
):
    """Plot DWBA and kinematical squared structure factors."""
    ax.semilogy(
        L, np.abs(F_kinematic) ** 2, "--", color="0.45",
        label=rf"kinematical, atten={crystal.atten:g}",
    )
    ax.semilogy(
        L, np.abs(F_dwba) ** 2, color="C0",
        label=dwba_label,
    )
    ax.set_xlabel(r"$L$ / r.l.u.")
    ax.set_ylabel(r"$|F|^2$ / electrons$^2$")
    ax.set_title(title)
    ax.grid(alpha=0.2, which="both")
    ax.legend()


figure, axes = plt.subplots(2, 2, figsize=(12.0, 8.0))
plot_factor_comparison(
    axes[0, 0], L_specular, F_dwba_total_specular,
    F_kinematic_specular, "specular (0, 0, L)",
    r"DWBA total field (equivalent $|F|^2$)",
)
plot_factor_comparison(
    axes[0, 1], L_rod, non_specular.F_contrast,
    F_kinematic_rod,
    rf"non-specular (0, 1, L), $\alpha_i=1.5\alpha_c$",
    r"DWBA contrast $|F_\Delta|^2$",
)

axes[1, 0].semilogy(
    L_specular, np.abs(specular.unperturbed_amplitude) ** 2,
    "--", color="0.45", label="optical reference",
)
axes[1, 0].semilogy(
    L_specular, np.abs(r_kinematic_specular) ** 2,
    "--", color="C2", label="kinematical field",
)
axes[1, 0].semilogy(
    L_specular, specular.reflectivity, color="C1",
    label=r"DWBA $|r_0+\delta r|^2$",
)
axes[1, 0].axvline(
    L_specular[np.argmin(np.abs(specular.prepared.alpha_i-alpha_c))],
    color="0.3", linestyle=":", label=r"$\alpha_c$",
)
axes[1, 0].set_xlabel(r"$L$ / r.l.u.")
axes[1, 0].set_ylabel("specular reflectivity")
axes[1, 0].set_title("coherent specular observable")
axes[1, 0].grid(alpha=0.2, which="both")
axes[1, 0].legend()

axes[1, 1].plot(
    L_rod, np.rad2deg(non_specular.prepared.alpha_i),
    label=r"$\alpha_i$",
)
axes[1, 1].plot(
    L_rod, np.rad2deg(non_specular.prepared.alpha_f),
    label=r"$\alpha_f$",
)
axes[1, 1].axhline(
    np.rad2deg(alpha_c), color="0.3", linestyle=":",
    label=r"$\alpha_c$",
)
axes[1, 1].set_xlabel(r"$L$ / r.l.u.")
axes[1, 1].set_ylabel("glancing angle / deg")
axes[1, 1].set_title("fixed-incidence z-mode geometry")
axes[1, 1].grid(alpha=0.2)
axes[1, 1].legend()

figure.suptitle(
    f"RuO2/TiO2 CTRs at {crystal.uc_bulk._E * 1e-3:g} keV"
)
figure.tight_layout()
figure